## configure runtime chain internals

- A configurable_fields method. This lets you configure particular fields of a runnable.This is related to the .bind method on runnables, but allows you to specify parameters for a given step in a chain at runtime rather than specifying them beforehand.
- A configurable_alternatives method. With this method, you can list out alternatives for any particular runnable that can be set during runtime, and swap them for those specified alternatives.

In [58]:
## configurable fields

import os
os.environ["OPENAI_MODEL_NAME"] = "deepseek-chat"
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com"
os.environ["OPENAI_API_KEY"] = "sk-762684b96deb4f748cb4383757f69a09"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField

model = ChatOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    model=os.environ["OPENAI_MODEL_NAME"],
    base_url=os.environ["OPENAI_API_BASE"],
    temperature=0
    ).configurable_fields(  # temperature=0，结果随机性为0； 
        temperature=ConfigurableField
            (   # temperature设置为可配置字段，id为llm_temperature，name为LLM Temperature，description为The temperature of the model；
            id="llm_temperature",
            name="LLM Temperature",
            description="The temperature of the model",
            )
        )

model.invoke("pick a random number")

AIMessage(content="Sure! Here's a random number for you: **42**. 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 7, 'total_tokens': 22, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-6e746cd5-1a4f-480f-97eb-c21cb16cac69-0', usage_metadata={'input_tokens': 7, 'output_tokens': 15, 'total_tokens': 22, 'input_token_details': {}, 'output_token_details': {}})

In [20]:
model.with_config(configurable={"llm_temperature": 2}).invoke("pick a random number") # model。with_config（configurable={"llm_temperature": 0.9}）设置temperature为0.9，结果随机性为0.9；

AIMessage(content="Okay, I've picked a random number! Here it is:\n\n**7**\n\nLet me know if you'd like me to pick another! 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 7, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-1a624cec-8740-40d7-a5c7-7f32afdcfe86-0', usage_metadata={'input_tokens': 7, 'output_tokens': 30, 'total_tokens': 37, 'input_token_details': {}, 'output_token_details': {}})

In [35]:
## just affect one step of the chain

prompt = PromptTemplate.from_template("pick a random number above {x}")
chain = prompt | model

chain.invoke({"x": 0})


AIMessage(content="Sure! Here's a random number above 0: **7**. 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 10, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-10e4729a-14dd-4ca3-92fa-8236b9a9e356-0', usage_metadata={'input_tokens': 10, 'output_tokens': 16, 'total_tokens': 26, 'input_token_details': {}, 'output_token_details': {}})

In [40]:
chain.with_config(configurable={"llm_temperature": 2}).invoke({"x": 0}) # chain.with_config(configurable={"llm_temperature": 0.9})设置temperature为0.9，结果随机性为0.9；

AIMessage(content="Okay, I just picked a random number above 0. Here it is:\n\n**7.3**\n\nLet me know if you'd like me to pick another! 😊", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 10, 'total_tokens': 45, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-376e200b-e44e-4eaa-b244-2a68a9d69b4c-0', usage_metadata={'input_tokens': 10, 'output_tokens': 35, 'total_tokens': 45, 'input_token_details': {}, 'output_token_details': {}})

## with HubRunnable

This is useful to allow for swithing of prompts.

In [48]:
import os

os.environ["LANGCHAIN_API_KEY"] = "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"

In [49]:
from langchain.runnables.hub import HubRunnable

prompt = HubRunnable("rlm/rag-prompt").configurable_fields(  # Langchain Hub是一个存储和共享提示词模版的远程共享平台，“rlm/rag-prompt”是其中一个专门为RAG设计的模版，接受question和context两个参数；生成适合RAG模型的提示。
    owner_repo_commit=ConfigurableField(
        id="hub_commit",
        name="Hub Commit",
        description="The Hub commit to pull from",
    )
)

prompt.invoke({"question": "foo", "context": "bar"})

/opt/anaconda3/envs/langchain_env/lib/python3.12/site-packages/langsmith/client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptValue(messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: foo \nContext: bar \nAnswer:", additional_kwargs={}, response_metadata={})])

In [59]:
prompt.with_config(configurable={"hub_commit": "rlm/rag-prompt-llama"}).invoke(
    {"question": "foo", "context": "bar"}
)

/opt/anaconda3/envs/langchain_env/lib/python3.12/site-packages/langsmith/client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


ChatPromptValue(messages=[HumanMessage(content="[INST]<<SYS>> You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.<</SYS>> \nQuestion: foo \nContext: bar \nAnswer: [/INST]", additional_kwargs={}, response_metadata={})])

## configurable alternatives
The configurable_alternatives() method allows us to swap out steps in a chain with an alternative. with this medthod, we can swap out one chat model for another。Below is a sample alternate between prompts.

In [60]:
prompt = PromptTemplate.from_template(
    "Tell me a joke about {topic}"
).configurable_alternatives(
    # This gives this field an id
    # When configuring the end runnable, we can then use this id to configure this field
    ConfigurableField(id="prompt"),
    # This sets a default_key.
    # If we specify this key, the default prompt (asking for a joke, as initialized above) will be used
    default_key="joke",
    # This adds a new option, with name `poem`
    poem=PromptTemplate.from_template("Write a short poem about {topic}"),
    # You can add more configuration options here
)
chain = prompt | model

# By default it will write a joke
chain.invoke({"topic": "bears"})

AIMessage(content="Sure! Here's a bear joke for you:\n\nWhy don’t bears wear shoes?  \nBecause they have bear feet! 🐾  \n\nHope that gave you a little chuckle! 😄", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 9, 'total_tokens': 48, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-8985d861-9f44-4db1-8f43-44403d6704a5-0', usage_metadata={'input_tokens': 9, 'output_tokens': 39, 'total_tokens': 48, 'input_token_details': {}, 'output_token_details': {}})

In [61]:
chain.with_config(configurable={"prompt": "poem"}).invoke({"topic": "bears"})

AIMessage(content="In forests deep where shadows play,  \nThe mighty bear begins its day.  \nThrough pines and streams, it roams with grace,  \nA gentle giant in nature's embrace.  \n\nIts fur, a cloak of earthy hue,  \nGuards against the morning dew.  \nWith paws that tread both soft and strong,  \nIt hums the forest's ancient song.  \n\nIn summer's warmth, it feasts and thrives,  \nOn berries sweet and honeyed hives.  \nWhen winter calls, it finds its den,  \nTo dream beneath the stars again.  \n\nOh bear, so wild, so wise, so free,  \nA symbol of the earth's decree.  \nIn your presence, we stand in awe,  \nOf nature's balance, pure and raw.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 9, 'total_tokens': 166, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_r

In [ ]:
both channge the llm and prompt

In [62]:
llm = ChatAnthropic(
    model="claude-3-haiku-20240307", temperature=0
).configurable_alternatives(
    # This gives this field an id
    # When configuring the end runnable, we can then use this id to configure this field
    ConfigurableField(id="llm"),
    # This sets a default_key.
    # If we specify this key, the default LLM (ChatAnthropic initialized above) will be used
    default_key="anthropic",
    # This adds a new option, with name `openai` that is equal to `ChatOpenAI()`
    openai=ChatOpenAI(),
    # This adds a new option, with name `gpt4` that is equal to `ChatOpenAI(model="gpt-4")`
    gpt4=ChatOpenAI(model="gpt-4"),
    # You can add more configuration options here
)
prompt = PromptTemplate.from_template(
    "Tell me a joke about {topic}"
).configurable_alternatives(
    # This gives this field an id
    # When configuring the end runnable, we can then use this id to configure this field
    ConfigurableField(id="prompt"),
    # This sets a default_key.
    # If we specify this key, the default prompt (asking for a joke, as initialized above) will be used
    default_key="joke",
    # This adds a new option, with name `poem`
    poem=PromptTemplate.from_template("Write a short poem about {topic}"),
    # You can add more configuration options here
)
chain = prompt | llm

# We can configure it write a poem with OpenAI
chain.with_config(configurable={"prompt": "poem", "llm": "openai"}).invoke(
    {"topic": "bears"}
)

NameError: name 'ChatAnthropic' is not defined

Save the configuration

In [ ]:
openai_joke = chain.with_config(configurable={"llm": "openai"})

openai_joke.invoke({"topic": "bears"})